# Notebook for trt selection development

In [1]:
import os
import pandas as pd
from pathlib import Path

import sqlalchemy as sa
import sqlalchemy.orm as so
import sys
import yaml


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "scoop").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not locate the SCOOP repo root.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CONFIG_ROOT = ROOT / "scoop" / "configuration"

In [2]:
from scoop.configuration import protocol_registry
from scoop.utils.hemonc_helpers import _load_hemonc_models
from scoop.utils.variant_selection import build_components_subquery, build_variant_query_artifacts #, execute_variant_query
from scoop.utils.treatment_selection import (
    get_siggroup_identifying_subquery, subquery_sigs_valid_groups, 
    get_base_sigs_df, subset_df_on_phase, 
    get_sig_component_stmt, categorise_components,
    )
from scoop.utils.treatment_containers import ProtocolGenerator

models = _load_hemonc_models()

In [3]:
REGISTRY_PATH = ROOT / "scoop" / "configuration" / "profiles" / "nsclc" / "nsclc_protocol_registry.yaml"

with REGISTRY_PATH.open("r", encoding="utf-8") as handle:
    raw = yaml.safe_load(handle)

In [4]:
raw["nsclc"].keys()

dict_keys(['version', 'conditions', 'source_phases', 'phase_combinations', 'protocols', 'entries'])

#### First layer - read phase sources, and combination data

In [5]:
default_conditions = raw["nsclc"]["conditions"]

In [6]:
default_conditions

['Lung cancer', 'Non-small cell lung cancer']

In [7]:
HEMONC_ENGINE_URL = os.environ["HEMONC_ENGINE_URL"]
def _require_engine_url(engine_url: str | None = None) -> str:
    resolved = engine_url or HEMONC_ENGINE_URL
    if not resolved:
        raise ValueError("Set HEMONC_ENGINE_URL or pass engine_url explicitly.")
    return resolved
engine = sa.create_engine(_require_engine_url(HEMONC_ENGINE_URL))

In [8]:
from scoop.utils.hemonc_helpers import condition_cuis_by_names

In [9]:
test_input = ["Lung cancer"]

In [10]:
with so.Session(engine) as session:
    print(condition_cuis_by_names(test_input, session))

[46104]


In [11]:
with so.Session(engine) as session:
    nsclc_registry = protocol_registry.RegistryYaml.from_yaml_path(REGISTRY_PATH, session)

In [12]:
nsclc_registry

RegistryYaml(label='nsclc', default_conditions=[46104, 642], version=1.0, source_phases={'surgery_only': SourcePhaseYaml(source_phase_id='surgery_only', condition_cuis=[46104, 642], component_requirements=ComponentRequirements(component_lists=[[]], category_restrictions={'chemotherapy': CategoryRestriction(type='exact', amount=0), 'radiotherapy': CategoryRestriction(type='exact', amount=0), 'immunotherapy': CategoryRestriction(type='exact', amount=0), 'surgery': CategoryRestriction(type='min', amount=1)}), study_restriction=None, curative_intent=True, regimens=None, type='definitive', standalone=True), 'port': SourcePhaseYaml(source_phase_id='port', condition_cuis=[46104, 642], component_requirements=ComponentRequirements(component_lists=[[]], category_restrictions={'chemotherapy': CategoryRestriction(type='exact', amount=0), 'radiotherapy': CategoryRestriction(type='min', amount=1), 'immunotherapy': CategoryRestriction(type='exact', amount=0), 'surgery': CategoryRestriction(type='exac

In [13]:
# with so.Session(engine) as session:
#     source_phases = {label: protocol_registry.SourcePhaseYaml.generate(
#                         source_phase, label, default_conditions, session) 
#                     for label, source_phase in raw["nsclc"]["source_phases"].items()}

In [14]:
nsclc_registry.source_phases

{'surgery_only': SourcePhaseYaml(source_phase_id='surgery_only', condition_cuis=[46104, 642], component_requirements=ComponentRequirements(component_lists=[[]], category_restrictions={'chemotherapy': CategoryRestriction(type='exact', amount=0), 'radiotherapy': CategoryRestriction(type='exact', amount=0), 'immunotherapy': CategoryRestriction(type='exact', amount=0), 'surgery': CategoryRestriction(type='min', amount=1)}), study_restriction=None, curative_intent=True, regimens=None, type='definitive', standalone=True),
 'port': SourcePhaseYaml(source_phase_id='port', condition_cuis=[46104, 642], component_requirements=ComponentRequirements(component_lists=[[]], category_restrictions={'chemotherapy': CategoryRestriction(type='exact', amount=0), 'radiotherapy': CategoryRestriction(type='min', amount=1), 'immunotherapy': CategoryRestriction(type='exact', amount=0), 'surgery': CategoryRestriction(type='exact', amount=0)}), study_restriction=None, curative_intent=True, regimens=['Radiation the

In [15]:
raw["nsclc"]["protocols"]["neoadjuvant_or_perioperative_ici_w_chemo"]

{'phases': ['neoadjuvant_chemo_ici', 'surgery_only', 'adjuvant_ici'],
 'optional_phases': {'adjuvant_ici': {'criteria': {'shared': 'component',
    'as_phase': 'neoadjuvant_chemo_ici'}}}}

In [16]:
protocols = {label: protocol_registry.ProtocolYaml.generate(label, protocol_data) 
             for label, protocol_data in raw["nsclc"]["protocols"].items()}

In [17]:
protocols

{'port_adjuvant_osimertinib': ProtocolYaml(protocol_id='port_adjuvant_osimertinib', source_phase_ids=['surgery_only', 'port', 'adj_osimertinib'], optional_phases=None),
 'surgery_then_port': ProtocolYaml(protocol_id='surgery_then_port', source_phase_ids=['surgery_only', 'port'], optional_phases=None),
 'neoadjuvant_or_perioperative_ici_w_chemo': ProtocolYaml(protocol_id='neoadjuvant_or_perioperative_ici_w_chemo', source_phase_ids=['neoadjuvant_chemo_ici', 'surgery_only', 'adjuvant_ici'], optional_phases={'adjuvant_ici': {'criteria': OptionalPhaseCriteria(shared='component', as_phase='neoadjuvant_chemo_ici')}}),
 'adjuvant_cisplatin_osimertinib': ProtocolYaml(protocol_id='adjuvant_cisplatin_osimertinib', source_phase_ids=['adjuvant_cisplatin_based', 'adjuvant_osimertinib'], optional_phases=None),
 'conc_crt_followed_by_durvalumab': ProtocolYaml(protocol_id='conc_crt_followed_by_durvalumab', source_phase_ids=['conc_crt', 'consolidation_durvalumab'], optional_phases=None),
 'conc_crt_foll

In [18]:
phase_unions = {
    label: protocol_registry.PhaseUnionYaml(phase_id=label, source_phase_ids=union_data["source_phases"])
    for label, union_data in raw["nsclc"]["phase_combinations"].items()
}

In [19]:
phase_unions

{'met_intervention': PhaseUnionYaml(phase_id='met_intervention', source_phase_ids=['met_tki', 'met_ptdb', 'met_ptdb_w_ici']),
 'ici_monotherapy_or_ici_with_chemotherapy_squamous': PhaseUnionYaml(phase_id='ici_monotherapy_or_ici_with_chemotherapy_squamous', source_phase_ids=['ici_monotherapy', 'ici_w_chemo_squamous']),
 'ici_monotherapy_or_ici_with_chemotherapy_nonsquamous': PhaseUnionYaml(phase_id='ici_monotherapy_or_ici_with_chemotherapy_nonsquamous', source_phase_ids=['ici_monotherapy', 'ici_w_chemo_nonsquamous']),
 'chemotherapy_or_atezolizumab_squamous': PhaseUnionYaml(phase_id='chemotherapy_or_atezolizumab_squamous', source_phase_ids=['ptdb_squamous', 'single_chemo_squamous', 'atezolizumab']),
 'chemotherapy_or_atezolizumab_nonsquamous': PhaseUnionYaml(phase_id='chemotherapy_or_atezolizumab_nonsquamous', source_phase_ids=['ptdb_nonsquamous', 'single_chemo_nonsquamous', 'atezolizumab'])}

#### 2nd layer - produce all protocols w/ categorised phases

In [20]:
nsclc_registry.source_phases["surgery_only"].categorise()

{'surgery'}

In [21]:
def find_retrievability(broad_categories):
    if "chemotherapy" in broad_categories or "immunotherapy" in broad_categories:
        return "variantable"
    elif "radiotherapy" in broad_categories:
        return "rt"
    elif "surgery" in broad_categories:
        return "surgery"
    else:
        return "null"

In [22]:
from scoop.utils import phase_resolution

In [23]:
eg_surgery_phase = nsclc_registry.source_phases["surgery_only"]

In [24]:
eg_surgery_phase

SourcePhaseYaml(source_phase_id='surgery_only', condition_cuis=[46104, 642], component_requirements=ComponentRequirements(component_lists=[[]], category_restrictions={'chemotherapy': CategoryRestriction(type='exact', amount=0), 'radiotherapy': CategoryRestriction(type='exact', amount=0), 'immunotherapy': CategoryRestriction(type='exact', amount=0), 'surgery': CategoryRestriction(type='min', amount=1)}), study_restriction=None, curative_intent=True, regimens=None, type='definitive', standalone=True)

In [25]:
phase_resolution.SurgeryPhase(phase_id=eg_surgery_phase.source_phase_id, phase_type=eg_surgery_phase.type, origin=eg_surgery_phase)

SurgeryPhase(phase_id='surgery_only', phase_type='definitive', origin=SourcePhaseYaml(source_phase_id='surgery_only', condition_cuis=[46104, 642], component_requirements=ComponentRequirements(component_lists=[[]], category_restrictions={'chemotherapy': CategoryRestriction(type='exact', amount=0), 'radiotherapy': CategoryRestriction(type='exact', amount=0), 'immunotherapy': CategoryRestriction(type='exact', amount=0), 'surgery': CategoryRestriction(type='min', amount=1)}), study_restriction=None, curative_intent=True, regimens=None, type='definitive', standalone=True))

In [26]:
eg_variantable_phase = nsclc_registry.source_phases["neoadjuvant_chemo_ici"]
eg_variantable_phase

SourcePhaseYaml(source_phase_id='neoadjuvant_chemo_ici', condition_cuis=[46104, 642], component_requirements=ComponentRequirements(component_lists=[['nivolumab', 'pembrolizumab', 'durvalumab', 'tislelizumab']], category_restrictions={'chemotherapy': CategoryRestriction(type='min', amount=1), 'radiotherapy': CategoryRestriction(type='exact', amount=0), 'immunotherapy': CategoryRestriction(type='exact', amount=1), 'surgery': CategoryRestriction(type='exact', amount=0)}), study_restriction=None, curative_intent=True, regimens=None, type='neoadjuvant', standalone=True)

In [27]:
# phase_resolution.VariantablePhase()

In [28]:
for phase in nsclc_registry.source_phases.values():
    print(phase.source_phase_id)
    cats = phase.categorise()
    print(cats)
    print(find_retrievability(cats))
    print()

surgery_only
{'surgery'}
surgery

port
{'radiotherapy'}
rt

adj_osimertinib
{'chemotherapy'}
variantable

neoadjuvant_chemo_ici
{'chemotherapy', 'immunotherapy'}
variantable

adjuvant_ici
{'chemotherapy'}
variantable

adjuvant_pembrolizumab
{'immunotherapy'}
variantable

adjuvant_atezolizumab_pembrolizumab
{'immunotherapy'}
variantable

adjuvant_osimertinib
{'chemotherapy'}
variantable

adjuvant_cisplatin_based
{'chemotherapy'}
variantable

adjuvant_optchemo_alectinib
{'chemotherapy'}
variantable

conc_crt
{'radiotherapy', 'chemotherapy'}
variantable

seq_crt_chemo
{'chemotherapy'}
variantable

definitive_rt
{'radiotherapy'}
rt

consolidation_durvalumab
{'immunotherapy'}
variantable

consolidation_osimertinib
{'chemotherapy'}
variantable

egfr_intervention
{'chemotherapy', 'immunotherapy'}
variantable

alk_intervention
{'chemotherapy'}
variantable

ros1_intervention
{'chemotherapy'}
variantable

braf_intervention
{'chemotherapy'}
variantable

ret_intervention
{'chemotherapy'}
variantab

In [29]:
build_variant_query_artifacts(nsclc_registry.source_phases["neoadjuvant_chemo_ici"].component_requirements, 
                              nsclc_registry.source_phases["neoadjuvant_chemo_ici"].condition_cuis)

VariantQueryArtifacts(components_subquery=<sqlalchemy.sql.selectable.Select object at 0x7f345e92ae40>, categorised_components_subquery=<sqlalchemy.sql.selectable.Subquery at 0x7f3460d5de50; anon_1>, variant_cuis_subquery=<sqlalchemy.sql.selectable.Subquery at 0x7f345e9ac180; anon_1>, variants_stmt=<sqlalchemy.sql.selectable.Select object at 0x7f345e92b110>)

In [30]:
condition_cuis = [46104, 642]

## Working on new RT pathway

In [31]:
rt_phases = []
for phase in nsclc_registry.source_phases.values():
    process_type = find_retrievability(phase.categorise())
    if process_type == "rt":
        rt_phases.append(phase)

In [32]:
for phase in rt_phases:
    print(phase)

source_phase_id='port' condition_cuis=[46104, 642] component_requirements=ComponentRequirements(component_lists=[[]], category_restrictions={'chemotherapy': CategoryRestriction(type='exact', amount=0), 'radiotherapy': CategoryRestriction(type='min', amount=1), 'immunotherapy': CategoryRestriction(type='exact', amount=0), 'surgery': CategoryRestriction(type='exact', amount=0)}) study_restriction=None curative_intent=True regimens=['Radiation therapy'] type='adjuvant' standalone=False
source_phase_id='definitive_rt' condition_cuis=[46104, 642] component_requirements=ComponentRequirements(component_lists=[[]], category_restrictions={'chemotherapy': CategoryRestriction(type='exact', amount=0), 'radiotherapy': CategoryRestriction(type='exact', amount=1), 'immunotherapy': CategoryRestriction(type='exact', amount=0), 'surgery': CategoryRestriction(type='exact', amount=0), 'supportive': CategoryRestriction(type='exact', amount=0)}) study_restriction=None curative_intent=True regimens=['Radiati

In [33]:
# if regimen, retrieve by regimen
# elif ComponentRequirements retrieve by component requirements
# else Exception

## Change of mind
# filter on both regimen and componentrequirements, making sure for both to be safe if permissive

#### Working on possible components replacement

In [34]:
with so.Session(engine) as session:
    original_test = pd.DataFrame(session.execute(sa.select(build_components_subquery(condition_cuis))).unique())

In [35]:
original_test

,variant_cui,component,component_cui,drug_inn,main_class,canmed_major_class,canmed_minor_class
0,141786,Paclitaxel,379,paclitaxel,Taxane,Microtubule inhibitor,Taxane
1,141786,Carboplatin,88,carboplatin,Platinum agent,Alkylating agent,Platinum agent
2,141786,Bevacizumab,68,bevacizumab,Anti-VEGFR antibody,Antineoplastic monoclonal antibody,Anti-VEGFR antibody
3,141786,Atezolizumab,49,atezolizumab,Anti-PD-L1 antibody,Immunostimulatory monoclonal antibody,Anti-PD-L1 antibody
4,141787,Paclitaxel,379,paclitaxel,Taxane,Microtubule inhibitor,Taxane
...,...,...,...,...,...,...,...
827,161285,Durvalumab,173,durvalumab,Anti-PD-L1 antibody,Immunostimulatory monoclonal antibody,Anti-PD-L1 antibody
828,161307,Sunvozertinib,161113,NaN,EGFR inhibitor,Tyrosine kinase inhibitor,EGFR inhibitor
829,161308,Taletrectinib,161120,NaN,ROS1 inhibitor,NaN,NaN
830,161311,Zongertinib,161243,NaN,ERBB2 inhibitor,Tyrosine kinase inhibitor,NaN


ISSUE: UNIMPORTANT
- sig of variant cui lists FEWER studies than variant
- variant in question: cisplatin monotherapy 131009	

In [36]:
def var_stmt_from_cuilist(cuilist: list):
    return sa.select(models.Variants).where(models.Variants.variant_cui.in_(cuilist))

In [37]:
with so.Session(engine) as session:
    missing_vars = session.execute(var_stmt_from_cuilist([131009])).unique().scalars().all()

In [38]:
# this is the reason for the difference:
# sig study items different to variant study items
# despite being monotherapy
# how can the variant have been studied in that context without the
# corresponding sig also being studied? 
for study_item in missing_vars[0].study_items:
    print(study_item.study_objects[0].study)
    print(study_item.study_objects[0].condition)

GEFCAPI 02
Carcinoma of unknown primary
Gatzemeier et al. 2000
Non-small cell lung cancer
Jacobs et al. 1992
Head and neck cancer


In [39]:
missing_vars[0].component_sigs[0].study

'GEFCAPI 02|Jacobs et al. 1992'

##### Old troubleshooting

130670 is from a breast cancer study....
and 130863 from a nasopharyngeal carcinoma study

In [40]:
missing_varinfo_stmt = (
    sa.select(models.Variants).where(models.Variants.variant_cui.in_([130670, 130863]))
)

In [41]:
with so.Session(engine) as session:
    missing_vars = session.execute(missing_varinfo_stmt).unique().scalars().all()

In [42]:
for var in missing_vars:
    print(f"{var.variant_cui}")

130670
130670
130670
130863
130863
130863
130863


In [43]:
missing_vars[-1].__dict__

{'_sa_instance_state': <sqlalchemy.orm.state.InstanceState at 0x7f345e57d4f0>,
 'allsigshavesequence': False,
 'date_tracer_modified': None,
 'timings': 0,
 'id': 23050,
 'allsigshavedose': True,
 'blob_version': 2,
 'fullyspecified': False,
 'allsigshavedoseunit': True,
 'branches': 0,
 'portions': 0,
 'variant': 'Variant #41',
 'allsigshaveduration': False,
 'components': 2,
 'regimen': 'Cisplatin and Gemcitabine (GC)',
 'variant_cui': 130863,
 'allsigshavedurationunit': False,
 'cyclesigs': 1,
 'regimen_cui': 2570,
 'version': 1,
 'allsigshavefrequency': True,
 'date_added': datetime.datetime(2025, 6, 22, 0, 0),
 'routes': 1,
 'allsigshaveroute': True,
 'date_study_modified': None,
 'sigs': 2,
 'allsigshavecyclesigs': True,
 'allsigshaveschedule': True,
 'temp': None,
 'study_items': [<hemonc_import.final_model.entities.variants_StudyMap at 0x7f345e54e040>,
 'blob_items': [<hemonc_import.final_model.entities.variants_BlobMap at 0x7f345e30e7b0>,
 'component_sigs': [<hemonc_import.fin

In [44]:
for sig in missing_vars[0].component_sigs:
    print(f"{sig.component} {sig.study}")

Cisplatin Fan et al. 2012|GLOB3|TAX 326
Docetaxel Fan et al. 2012|GLOB3|TAX 326


In [45]:
missing_vars[0].study_items[0].study_objects[0].__dict__ # Breast cancer study????

{'_sa_instance_state': <sqlalchemy.orm.state.InstanceState at 0x7f345e380530>,
 'condition_cui': 572.0,
 'registry': None,
 'trial_id': None,
 'date_added': datetime.datetime(2019, 5, 27, 0, 0),
 'sact': True,
 'unreg_study': False,
 'end': 'Not reported',
 'sponsor_type': <Studies_Sponsor_typeEnum.GOVERNMENT: 'government'>,
 'biomarker': 'Triple-negative',
 'enrollment': 'Not reported',
 'start': 'Not reported',
 'phase': 'Phase 3',
 'study': 'Fan et al. 2012',
 'protocol': False,
 'study_design': <Studies_Study_designEnum.IN_TO_CLASS_SWITCH: 'in-class switch'>,
 'id': 2486,
 'pubs_in_hemonc': 1,
 'study_design_imputed': False,
 'condition': 'Breast cancer',
 'reg_study': False,
 'temp': None,
 'sponsor_items': [<hemonc_import.final_model.entities.studies_SponsorMap at 0x7f345e394640>],
 'study_group_items': [],
 'condition_obj': <hemonc_import.final_model.entities.Conditions at 0x7f345e394770>,
 'condition_cui_obj': <hemonc_import.final_model.entities.Conditions at 0x7f345e394770>}

In [46]:
for study_item in missing_vars[0].study_items:
    print(study_item.study)
    for study_object in study_item.study_objects:
        print(study_object.condition)
        print(study_object.condition_cui)

Fan et al. 2012
Breast cancer
572.0


In [47]:
for sig in missing_vars[-1].component_sigs:
    print(f"{sig.component} {sig.study}")

Cisplatin GEM20110714|JUPITER-02|WJTOG0203|Liu et al. 2024
Gemcitabine GEM20110714|JUPITER-02|WJTOG0203|Liu et al. 2024


In [48]:
for study_item in missing_vars[-1].study_items:
    print(study_item.study)
    for study_object in study_item.study_objects:
        print(study_object.condition)
        print(study_object.condition_cui)

CAPTAIN-1st
Nasopharyngeal carcinoma
639.0
GEM20110714
Nasopharyngeal carcinoma
639.0
JUPITER-02
Nasopharyngeal carcinoma
639.0
RATIONALE 309
Nasopharyngeal carcinoma
639.0


In [49]:
missing_vars[0].__dict__

{'_sa_instance_state': <sqlalchemy.orm.state.InstanceState at 0x7f345e57cef0>,
 'allsigshavesequence': False,
 'date_tracer_modified': datetime.datetime(2025, 3, 25, 0, 0),
 'timings': 0,
 'id': 4182,
 'allsigshavedose': True,
 'blob_version': 2,
 'fullyspecified': False,
 'allsigshavedoseunit': True,
 'branches': 0,
 'portions': 0,
 'variant': 'Variant #01',
 'allsigshaveduration': False,
 'components': 2,
 'regimen': 'Cisplatin and Docetaxel (DC)',
 'variant_cui': 130670,
 'allsigshavedurationunit': False,
 'cyclesigs': 1,
 'regimen_cui': 2569,
 'version': 1,
 'allsigshavefrequency': True,
 'date_added': datetime.datetime(2024, 10, 18, 0, 0),
 'routes': 1,
 'allsigshaveroute': True,
 'date_study_modified': None,
 'sigs': 2,
 'allsigshavecyclesigs': True,
 'allsigshaveschedule': True,
 'temp': None,
 'study_items': [<hemonc_import.final_model.entities.variants_StudyMap at 0x7f345e432050>],
 'blob_items': [<hemonc_import.final_model.entities.variants_BlobMap at 0x7f345e57e330>,
 'compo

In [50]:
# with so.Session(engine) as session:
#     stmt_full = (
#         sa.select(models.Variants)
#         .join(subquery, models.Variants.variant_cui == subquery.c.variant_cui)
#     )
#     chosen_variants = session.execute(stmt_full).unique().scalars().all()

## Working on ~~semi-~~integrated pathway

In [51]:
components = get_sig_component_stmt(condition_cuis, models)
categorised = categorise_components(components)

In [52]:
components

In [53]:
with so.Session(engine) as session:
    display(pd.DataFrame(session.execute(sa.select(categorised))))

,variant_cui,canmed_major_class,canmed_minor_class,main_class,component,component_cui,study,phase,broad_category
0,129610.0,NaN,NaN,KRAS inhibitor,Adagrasib,45753.0,KRYSTAL-1_CRC|KRYSTAL-1_NSCLC,NaN,chemotherapy
1,129627.0,Microtubule inhibitor,Taxane,Taxane,Paclitaxel,379.0,LUX-Lung 5,NaN,chemotherapy
2,129627.0,Tyrosine kinase inhibitor,EGFR inhibitor,EGFR inhibitor,Afatinib,13.0,LUX-Lung 5,NaN,chemotherapy
3,129628.0,Tyrosine kinase inhibitor,EGFR inhibitor,EGFR inhibitor,Afatinib,13.0,LUX-Head and Neck 1|LUX-Head and Neck 3|LUX-Lu...,NaN,chemotherapy
4,129629.0,Tyrosine kinase inhibitor,EGFR inhibitor,EGFR inhibitor,Afatinib,13.0,LUX-Lung 2|LUX-Lung 3|LUX-Lung 6|LUX-Lung 7,NaN,chemotherapy
...,...,...,...,...,...,...,...,...,...
746,NaN,NaN,NaN,NaN,External beam radiotherapy,1608.0,Neuhaus et al. 2009|ENTER,NaN,radiotherapy
747,NaN,NaN,NaN,NaN,Granulocyte colony-stimulating factor,45734.0,ARCTIC|E7389-G000-302|TAX 320,NaN,supportive
748,NaN,NaN,NaN,NaN,Steroid,45523.0,BEYOND|CA184-041_NSCLC|CA184-104|Stathopoulos ...,NaN,supportive
749,NaN,NaN,NaN,NaN,Steroid,45523.0,CA184-041_NSCLC|CA184-041_SCLC,NaN,supportive


In [54]:
# eg_comp_reqs = nsclc_registry.source_phases["neoadjuvant_chemo_ici"].component_requirements

In [55]:
eg_source_phase = nsclc_registry.source_phases["adjuvant_optchemo_alectinib"]

In [56]:
eg_comp_reqs = eg_source_phase.component_requirements

In [57]:
phase_type = eg_source_phase.type.title()

In [58]:

valid_groups_subquery = get_siggroup_identifying_subquery(eg_comp_reqs, categorised)
sigs_for_valid_groups_subquery= subquery_sigs_valid_groups(valid_groups_subquery, models)
stmt = sa.select(sigs_for_valid_groups_subquery, models.Sigs).join(models.Sigs, models.Sigs.id==sigs_for_valid_groups_subquery.c.id) #.filter(or_for_piped_col(sigs_for_valid_groups_subquery.c.phase, phase_type.title()))


In [59]:
with so.Session(engine) as session:
    new_cat_test = pd.DataFrame(session.execute(stmt))

In [60]:
eg_comp_reqs

ComponentRequirements(component_lists=[['alectinib']], category_restrictions={'chemotherapy': CategoryRestriction(type='min', amount=1), 'radiotherapy': CategoryRestriction(type='exact', amount=0), 'immunotherapy': CategoryRestriction(type='exact', amount=0), 'surgery': CategoryRestriction(type='exact', amount=0)})

Note variant cui 131619 is for dexamethasone and wbrt

woooo :) ready to populate treatment phase

In [61]:
new_cat_test

,single_study,matched_phase,matched_variant_cui,variant_cui,phase,study,component,id,regimen,variant,Sigs
0,AF-001JP,None,129634.0,129634.0,None,AF-001JP|ALEX|J-ALEX,Alectinib,768,Alectinib monotherapy,Variant #01,<hemonc_import.final_model.entities.Sigs objec...
1,ALEX,None,129634.0,129634.0,None,AF-001JP|ALEX|J-ALEX,Alectinib,768,Alectinib monotherapy,Variant #01,<hemonc_import.final_model.entities.Sigs objec...
2,J-ALEX,None,129634.0,129634.0,None,AF-001JP|ALEX|J-ALEX,Alectinib,768,Alectinib monotherapy,Variant #01,<hemonc_import.final_model.entities.Sigs objec...
3,ALESIA,None,129635.0,129635.0,None,ALESIA|ALEX,Alectinib,769,Alectinib monotherapy,Variant #02,<hemonc_import.final_model.entities.Sigs objec...
4,ALEX,None,129635.0,129635.0,None,ALESIA|ALEX,Alectinib,769,Alectinib monotherapy,Variant #02,<hemonc_import.final_model.entities.Sigs objec...
5,ALINA,None,141790.0,141790.0,None,ALINA,Alectinib,771,Alectinib monotherapy,Variant #04,<hemonc_import.final_model.entities.Sigs objec...


In [62]:
new_cat_test.fillna("None").groupby(["phase", "variant_cui", "study", "regimen"], sort=False)["Sigs"].agg(list).to_dict()

{('None',
  129634.0,
  'AF-001JP|ALEX|J-ALEX',
  'Alectinib monotherapy'): [<hemonc_import.final_model.entities.Sigs at 0x7f345e454f50>, <hemonc_import.final_model.entities.Sigs at 0x7f345e454f50>, <hemonc_import.final_model.entities.Sigs at 0x7f345e454f50>],
 ('None',
  129635.0,
  'ALESIA|ALEX',
  'Alectinib monotherapy'): [<hemonc_import.final_model.entities.Sigs at 0x7f345e4579d0>, <hemonc_import.final_model.entities.Sigs at 0x7f345e4579d0>],
 ('None',
  141790.0,
  'ALINA',
  'Alectinib monotherapy'): [<hemonc_import.final_model.entities.Sigs at 0x7f345e457610>]}

In [63]:
new_cat_test.fillna("None").groupby(["phase", "variant_cui", "study"], sort=False)["Sigs"].agg(list).to_list()

[[<hemonc_import.final_model.entities.Sigs at 0x7f345e454f50>,

In [64]:
new_cat_test.groupby(["phase", "variant_cui", "study"], sort=False)["Sigs"].apply(list)

Series([], Name: Sigs, dtype: object)

In [65]:
new_cat_test.fillna("None").groupby(["phase", "variant_cui", "study"], sort=False)["Sigs"].agg(list).to_list()

[[<hemonc_import.final_model.entities.Sigs at 0x7f345e454f50>,

In [66]:
new_cat_test # woooo - var test and non var test passed

,single_study,matched_phase,matched_variant_cui,variant_cui,phase,study,component,id,regimen,variant,Sigs
0,AF-001JP,None,129634.0,129634.0,None,AF-001JP|ALEX|J-ALEX,Alectinib,768,Alectinib monotherapy,Variant #01,<hemonc_import.final_model.entities.Sigs objec...
1,ALEX,None,129634.0,129634.0,None,AF-001JP|ALEX|J-ALEX,Alectinib,768,Alectinib monotherapy,Variant #01,<hemonc_import.final_model.entities.Sigs objec...
2,J-ALEX,None,129634.0,129634.0,None,AF-001JP|ALEX|J-ALEX,Alectinib,768,Alectinib monotherapy,Variant #01,<hemonc_import.final_model.entities.Sigs objec...
3,ALESIA,None,129635.0,129635.0,None,ALESIA|ALEX,Alectinib,769,Alectinib monotherapy,Variant #02,<hemonc_import.final_model.entities.Sigs objec...
4,ALEX,None,129635.0,129635.0,None,ALESIA|ALEX,Alectinib,769,Alectinib monotherapy,Variant #02,<hemonc_import.final_model.entities.Sigs objec...
5,ALINA,None,141790.0,141790.0,None,ALINA,Alectinib,771,Alectinib monotherapy,Variant #04,<hemonc_import.final_model.entities.Sigs objec...


In [67]:
len(new_cat_test)

6

In [68]:
# fix source phase conditions DONE
# add in-statement study filtering

# add post-df study filtering
# add post-df phase filtering DONE
# add post-df context filtering

In [69]:
eg_source_phase = nsclc_registry.source_phases["definitive_rt"]

In [70]:
with so.Session(engine) as session:
    test_base_df = get_base_sigs_df(eg_source_phase, session)

In [71]:
subset_df_on_phase(test_base_df, eg_source_phase.type)

,single_study,matched_phase,matched_variant_cui,variant_cui,phase,study,component,id,regimen,variant,Sigs
1,Ball et al. 1997,Definitive,None,None,Definitive,Ball et al. 1997,External beam radiotherapy,21487,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...
2,CALGB 9130,Definitive,None,None,Definitive,CALGB 9130|Groen et al. 2004|JCOG0301|Morton e...,External beam radiotherapy,21489,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...
3,Groen et al. 2004,Definitive,None,None,Definitive,CALGB 9130|Groen et al. 2004|JCOG0301|Morton e...,External beam radiotherapy,21489,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...
4,JCOG0301,Definitive,None,None,Definitive,CALGB 9130|Groen et al. 2004|JCOG0301|Morton e...,External beam radiotherapy,21489,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...
5,Morton et al. 1991,Definitive,None,None,Definitive,CALGB 9130|Groen et al. 2004|JCOG0301|Morton e...,External beam radiotherapy,21489,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...
6,Perez et al. 1988,Definitive,None,None,Definitive,CALGB 9130|Groen et al. 2004|JCOG0301|Morton e...,External beam radiotherapy,21489,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...
7,RTOG 93-04,Definitive,None,None,Definitive,CALGB 9130|Groen et al. 2004|JCOG0301|Morton e...,External beam radiotherapy,21489,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...
8,LAMP,Definitive,None,None,Definitive,LAMP|RTOG 9410,External beam radiotherapy,21499,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...
9,RTOG 9410,Definitive,None,None,Definitive,LAMP|RTOG 9410,External beam radiotherapy,21499,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...
10,LAMP,Definitive,None,None,Definitive,LAMP|RTOG 9410,External beam radiotherapy,21504,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...


In [72]:
test_base_df.drop_duplicates(subset=["id"])

,single_study,matched_phase,matched_variant_cui,variant_cui,phase,study,component,id,regimen,variant,Sigs
0,ECOG E3590,Adjuvant,None,None,Adjuvant,ECOG E3590,External beam radiotherapy,21455,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...
1,Ball et al. 1997,Definitive,None,None,Definitive,Ball et al. 1997,External beam radiotherapy,21487,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...
2,CALGB 9130,Definitive,None,None,Definitive,CALGB 9130|Groen et al. 2004|JCOG0301|Morton e...,External beam radiotherapy,21489,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...
8,LAMP,Definitive,None,None,Definitive,LAMP|RTOG 9410,External beam radiotherapy,21499,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...
10,LAMP,Definitive,None,None,Definitive,LAMP|RTOG 9410,External beam radiotherapy,21504,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...


In [73]:
# Worked :)
# dfs = dict()
# for phase in source_phases.values():
#     with so.Session(engine) as session:
#         test_base_df = get_base_sigs_df(phase, session)
#         dfs[phase.source_phase_id] = subset_df_on_phase(test_base_df, phase.type)

In [74]:
from scoop.utils.treatment_selection import source_phase_to_sigdict

In [75]:
# Worked :) 
# for phase in source_phases.values():
#     with so.Session(engine) as session:
#         sigdict = source_phase_to_sigdict(phase, session)
#         print(f"{phase.source_phase_id}: len: {len(sigdict)}")

In [76]:
with so.Session(engine) as session:
    eg_sigdict = source_phase_to_sigdict(nsclc_registry.source_phases["neoadjuvant_chemo_ici"], session)

In [77]:
# Need to check these: non curative is a property of the STUDY not of the sigs
# Correction: per hemonckb data dictionary study context is a property of the study-regimen dyad
# Will need to think about implications of this but not immediate priority

# FIGURED IT OUT:
# This is happenning because supportive components are not assigned a variant CUI
# Thereby cleaving them from the primary components they match with regimen and study on
# Therefore even though the regimen might be e.g. cisplatin and pemetrexed
# These supportive components were not associated with those chemos in the grouping process
# dfs["best_supportive_care"]

In [78]:
nsclc_registry.source_phases["platinum_doublet_3rd_gen_agent"]

SourcePhaseYaml(source_phase_id='platinum_doublet_3rd_gen_agent', condition_cuis=[46104, 642], component_requirements=ComponentRequirements(component_lists=[['Platinum agent'], ['taxane', 'gemcitabine', 'vinorelbine']], category_restrictions={'chemotherapy': CategoryRestriction(type='exact', amount=2), 'radiotherapy': CategoryRestriction(type='exact', amount=0), 'immunotherapy': CategoryRestriction(type='exact', amount=0), 'surgery': CategoryRestriction(type='exact', amount=0)}), study_restriction=None, curative_intent=True, regimens=None, type='None', standalone=True)

## Working on protocols and combined phases

In [79]:
eg_protocol = protocols["neoadjuvant_or_perioperative_ici_w_chemo"]

In [80]:
eg_protocol

ProtocolYaml(protocol_id='neoadjuvant_or_perioperative_ici_w_chemo', source_phase_ids=['neoadjuvant_chemo_ici', 'surgery_only', 'adjuvant_ici'], optional_phases={'adjuvant_ici': {'criteria': OptionalPhaseCriteria(shared='component', as_phase='neoadjuvant_chemo_ici')}})

In [81]:
phase_unions

{'met_intervention': PhaseUnionYaml(phase_id='met_intervention', source_phase_ids=['met_tki', 'met_ptdb', 'met_ptdb_w_ici']),
 'ici_monotherapy_or_ici_with_chemotherapy_squamous': PhaseUnionYaml(phase_id='ici_monotherapy_or_ici_with_chemotherapy_squamous', source_phase_ids=['ici_monotherapy', 'ici_w_chemo_squamous']),
 'ici_monotherapy_or_ici_with_chemotherapy_nonsquamous': PhaseUnionYaml(phase_id='ici_monotherapy_or_ici_with_chemotherapy_nonsquamous', source_phase_ids=['ici_monotherapy', 'ici_w_chemo_nonsquamous']),
 'chemotherapy_or_atezolizumab_squamous': PhaseUnionYaml(phase_id='chemotherapy_or_atezolizumab_squamous', source_phase_ids=['ptdb_squamous', 'single_chemo_squamous', 'atezolizumab']),
 'chemotherapy_or_atezolizumab_nonsquamous': PhaseUnionYaml(phase_id='chemotherapy_or_atezolizumab_nonsquamous', source_phase_ids=['ptdb_nonsquamous', 'single_chemo_nonsquamous', 'atezolizumab'])}

In [82]:
eg_protocol = protocols["neoadjuvant_or_perioperative_ici_w_chemo"]

In [83]:
protocol_generator = ProtocolGenerator(source_phases=nsclc_registry.source_phases, phase_unions=phase_unions)

In [84]:
with so.Session(engine) as session:
    test_sb_lists_by_phase = protocol_generator.get_source_phases_sb_list(eg_protocol, session)

In [85]:
with so.Session(engine) as session:
    test_protocols = protocol_generator.get_protocols_from_protocol_yaml(eg_protocol, session)

In [86]:
with so.Session(engine) as session:
    test_singleton_protocol = protocol_generator.get_protocol_from_source_phase(
        "neoadjuvant_chemo_ici", session
        )

In [87]:
test_protocols[0]

Treatment Protocol: neoadjuvant_or_perioperative_ici_w_chemo (3 phase(s))

In [88]:
test_protocols[1]

Treatment Protocol: neoadjuvant_or_perioperative_ici_w_chemo (2 phase(s))

In [89]:
test_singleton_protocol[0]

Treatment Protocol: neoadjuvant_chemo_ici (1 phase(s))

In [90]:
test_protocols[0]

Treatment Protocol: neoadjuvant_or_perioperative_ici_w_chemo (3 phase(s))

### Further subsetting of components

In [91]:
nsclc_registry.source_phases["seq_crt_chemo"]

SourcePhaseYaml(source_phase_id='seq_crt_chemo', condition_cuis=[46104, 642], component_requirements=ComponentRequirements(component_lists=[[]], category_restrictions={'chemotherapy': CategoryRestriction(type='min', amount=1), 'radiotherapy': CategoryRestriction(type='exact', amount=0), 'immunotherapy': CategoryRestriction(type='exact', amount=0), 'surgery': CategoryRestriction(type='exact', amount=0)}), study_restriction='must_have_rt', curative_intent=True, regimens=None, type=None, standalone=False)

In [92]:
with so.Session(engine) as session:
    test_seq_crt = protocol_generator.get_protocols_from_protocol_yaml(protocols["seq_crt"], session)

In [93]:
# chemo maybe overly restrictive but fine for now - better than before
# low single dose in first rt option should be investigated - but not priority
test_seq_crt[0]

Treatment Protocol: seq_crt (2 phase(s))

In [94]:
with so.Session(engine) as session:
    test_port = protocol_generator.get_protocols_from_protocol_yaml(
        protocols["port_adjuvant_osimertinib"], session
        )

In [95]:
test_port[0]

Treatment Protocol: port_adjuvant_osimertinib (3 phase(s))

In [96]:
with so.Session(engine) as session:
    test_surg_port = protocol_generator.get_protocols_from_protocol_yaml(
        protocols["surgery_then_port"], session
        )

In [97]:
test_surg_port[0]

Treatment Protocol: surgery_then_port (2 phase(s))

## ...